# 📚 Smart PDF to Text Extractor — Optimized for Colab T4

**Pipeline:** Ingestion & Routing → Native Extract → Scanned OCR → Post-processing → Output

**Cải tiến so với phiên bản cũ:**
- ✅ Tương thích PaddleOCR 2.x **và** 3.x (không còn crash)
- ✅ Checkpoint / resume — ngắt phiên không mất dữ liệu
- ✅ RAM-safe: xử lý từng ảnh rồi xóa, `gc.collect()` sau mỗi file
- ✅ Bước 4 Post-processing đầy đủ: regex, line-unbreak, chuẩn UTF-8
- ✅ OCR tiếng Việt (`lang='vi'`) + fallback latin / tesseract
- ✅ Tất cả `pip install` gộp 1 cell duy nhất

⚠️ **Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`

## 0️⃣ Cài đặt dependencies (chỉ chạy 1 lần / runtime)

In [1]:
# ── Cài hết một lần, tránh restart nhiều lần làm mất RAM ──────────────────
import subprocess, sys

def run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# System packages
run('apt-get -qq update && apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-vie tesseract-ocr-eng 2>/dev/null')

# Python packages — version-pinned để tránh conflict
run(f'{sys.executable} -m pip install -q --upgrade pip')
run(f'{sys.executable} -m pip install -q PyMuPDF pdfplumber langdetect pdf2image')
run(f'{sys.executable} -m pip install -q paddlepaddle-gpu paddleocr')
run(f'{sys.executable} -m pip install -q pytesseract Pillow numpy')

print('✅ Cài xong toàn bộ dependencies.')

✅ Cài xong toàn bộ dependencies.


## 1️⃣ Mount Google Drive & Cấu hình

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ══════════════════════════════════════════════════════
#  ⚙️  CẤU HÌNH — chỉnh đường dẫn nếu cần
# ══════════════════════════════════════════════════════
DRIVE_BASE    = '/content/drive/MyDrive'
BOOKS_DIR     = f'{DRIVE_BASE}/book'
PROCESSED_DIR = f'{DRIVE_BASE}/book/processed'
BOOKS_OUT     = f'{PROCESSED_DIR}/books'

# OCR settings
OCR_DPI           = 300       # 250 DPI: cân bằng tốc độ vs chất lượng cho T4
OCR_BATCH_PAGES   = 6        # Xử lý 2 trang/lần → an toàn RAM (tăng lên 4 nếu GPU còn nhiều VRAM)
OCR_LANG_PRIMARY  = 'vi'      # Tiếng Việt — PaddleOCR hỗ trợ tốt dấu tiếng Việt
OCR_LANG_FALLBACK = 'latin'   # Fallback nếu 'vi' không load được
TESSERACT_LANG    = 'vie+eng' # Dùng khi PaddleOCR thất bại hoàn toàn
MAX_PAGES_PER_FILE = None      # None = không giới hạn; đặt số nguyên để debug nhanh
SKIP_IF_TXT_EXISTS = True      # True = resume được nếu ngắt phiên
# ══════════════════════════════════════════════════════

for p in [PROCESSED_DIR, BOOKS_OUT]:
    os.makedirs(p, exist_ok=True)

if not os.path.isdir(BOOKS_DIR):
    raise FileNotFoundError(f'Không tìm thấy thư mục PDF: {BOOKS_DIR}')

pdf_files = sorted([f for f in os.listdir(BOOKS_DIR) if f.lower().endswith('.pdf')])
print(f'📂 Nguồn PDF  : {BOOKS_DIR}')
print(f'💾 Lưu output : {BOOKS_OUT}')
print(f'📄 Tìm thấy   : {len(pdf_files)} file PDF')

Mounted at /content/drive
📂 Nguồn PDF  : /content/drive/MyDrive/book
💾 Lưu output : /content/drive/MyDrive/book/processed/books
📄 Tìm thấy   : 7 file PDF


## 🔧 Utility — Các hàm dùng chung

In [3]:
import re
import gc
import json
import time
import unicodedata
from langdetect import detect, LangDetectException

# ── Phát hiện ngôn ngữ ───────────────────────────────────────────────────
def detect_lang(text: str) -> str:
    """Trả về mã ngôn ngữ, mặc định 'vi' nếu không rõ."""
    if not text or not text.strip():
        return 'vi'
    try:
        return detect(text[:3000])
    except Exception:
        return 'vi'

# ── Checkpoint helpers ────────────────────────────────────────────────────
CHECKPOINT_FILE = os.path.join(PROCESSED_DIR, 'checkpoint.json')

def load_checkpoint() -> dict:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(data: dict):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

checkpoint = load_checkpoint()
print(f'📋 Checkpoint: {len(checkpoint)} file đã xử lý trước đó.')

# ── Ghi output file ───────────────────────────────────────────────────────
def write_output(pdf_filename: str, content: str, parser: str, lang: str) -> str:
    stem = Path(pdf_filename).stem
    out_path = os.path.join(BOOKS_OUT, f'{stem}.txt')
    header = (
        f'# SOURCE: {pdf_filename}\n'
        f'# TIER: tier1\n'
        f'# LANGUAGE: {lang}\n'
        f'# DOC_TYPE: book\n'
        f'# PARSER: {parser}\n'
        f'# {"-" * 60}\n\n'
    )
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(header + content)
    return out_path

print('✅ Utility functions sẵn sàng.')

📋 Checkpoint: 2 file đã xử lý trước đó.
✅ Utility functions sẵn sàng.


## 2️⃣ Bước 1 — Ingestion & Routing

Dùng PyMuPDF để đo **text density** của 5 trang đầu.
- `avg_chars/page < 120` **hoặc** `non_empty_ratio < 0.30` → `scanned_ocr`
- Còn lại → `native_extract`

In [4]:
import fitz  # PyMuPDF
from statistics import mean

MIN_CHARS_PER_PAGE  = 120
MIN_NON_EMPTY_RATIO = 0.30

def classify_pdf(pdf_path: str, sample: int = 5) -> tuple[str, dict]:
    doc = fitz.open(pdf_path)
    total = len(doc)
    n = min(sample, total)
    chars = [len((doc[i].get_text('text') or '').strip()) for i in range(n)]
    doc.close()

    avg   = mean(chars) if chars else 0.0
    ratio = sum(1 for c in chars if c > 0) / n if n else 0.0
    route = 'scanned_ocr' if (avg < MIN_CHARS_PER_PAGE or ratio < MIN_NON_EMPTY_RATIO) else 'native_extract'
    return route, {'total_pages': total, 'avg_chars': round(avg, 1), 'non_empty_ratio': round(ratio, 3)}

native_pdfs, scanned_pdfs, routing_report = [], [], []

for pdf in pdf_files:
    # Nếu đã xử lý xong → bỏ qua routing cũng được, nhưng vẫn phân loại để in report
    route, stats = classify_pdf(os.path.join(BOOKS_DIR, pdf))
    routing_report.append({'file': pdf, 'route': route, **stats})
    (native_pdfs if route == 'native_extract' else scanned_pdfs).append(pdf)

# Lưu routing report
with open(os.path.join(PROCESSED_DIR, 'routing_report.json'), 'w', encoding='utf-8') as f:
    json.dump(routing_report, f, ensure_ascii=False, indent=2)

print('=' * 55)
print('BƯỚC 1 — ROUTING REPORT')
print('=' * 55)
print(f'Tổng PDF   : {len(pdf_files)}')
print(f'Native     : {len(native_pdfs)}')
print(f'Scanned OCR: {len(scanned_pdfs)}')
print()
for r in routing_report:
    flag = '🖨️ ' if r['route'] == 'scanned_ocr' else '📝'
    print(f"{flag} {r['file']} | avg_chars={r['avg_chars']} | non_empty={r['non_empty_ratio']}")

BƯỚC 1 — ROUTING REPORT
Tổng PDF   : 7
Native     : 2
Scanned OCR: 5

📝 623781605-The-Muscle-and-Strength-Training-Pyramid-v2-0-Training-by-Eric-Helms-Z-lib-org-1.pdf | avg_chars=2437.6 | non_empty=1.0
🖨️  ACSM Guidelines for Exercise Testing and Prescription.pdf | avg_chars=0 | non_empty=0.0
🖨️  Back Mechanic - Stuart McGill - Director's Cut Edition-1.pdf | avg_chars=0 | non_empty=0.0
🖨️  Science and Development of Muscle Hypertrophy — Brad Schoenfeld.pdf | avg_chars=0 | non_empty=0.0
🖨️  Starting Strength — Mark Rippetoe.pdf | avg_chars=0 | non_empty=0.0
🖨️  The Muscle and Strength Pyramid 1 .pdf | avg_chars=0 | non_empty=0.0
📝 The Muscle and Strength Pyramid.pdf | avg_chars=2161.2 | non_empty=1.0


## 3️⃣ Bước 2 — Native PDF Extraction

Dùng PyMuPDF để bóc text từ PDF có text layer.
Sắp xếp block theo tọa độ (top→down, left→right) để giữ đúng thứ tự đọc.

In [5]:
import fitz

def extract_native(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages_out = []

    for page in doc:
        # get_text('blocks') trả về (x0,y0,x1,y1,text,block_no,block_type)
        blocks = page.get_text('blocks') or []
        # Sắp xếp: trên→dưới, trái→phải (làm tròn y để handle cùng dòng)
        blocks.sort(key=lambda b: (round(b[1] / 5) * 5, b[0]))
        lines = []
        for b in blocks:
            txt = (b[4] or '').strip()
            if not txt:
                continue
            cleaned = [ln.strip() for ln in txt.splitlines() if len(ln.strip()) >= 2]
            if cleaned:
                lines.append('\n'.join(cleaned))
        page_text = '\n'.join(lines).strip()
        if page_text:
            pages_out.append(page_text)

    doc.close()
    return '\n\n'.join(pages_out)

# ── Xử lý ────────────────────────────────────────────────────────────────
native_ok, native_fail = 0, []
print('=' * 55)
print(f'BƯỚC 2 — NATIVE EXTRACT ({len(native_pdfs)} file)')
print('=' * 55)

for pdf in native_pdfs:
    out_path = os.path.join(BOOKS_OUT, f'{Path(pdf).stem}.txt')

    # Resume: bỏ qua nếu đã có output
    if SKIP_IF_TXT_EXISTS and os.path.isfile(out_path):
        print(f'⏭️  SKIP {pdf} (đã có output)')
        native_ok += 1
        continue

    t0 = time.time()
    try:
        raw = extract_native(os.path.join(BOOKS_DIR, pdf))
        if not raw.strip():
            raise ValueError('Không extract được text')
        lang = detect_lang(raw)
        # Post-processing nhẹ (Step 4 đầy đủ ở cell riêng)
        raw = re.sub(r'[ \t]+', ' ', raw)
        raw = re.sub(r'\n{3,}', '\n\n', raw)
        write_output(pdf, raw, 'Native (PyMuPDF)', lang)
        checkpoint[pdf] = {'status': 'native_ok', 'chars': len(raw)}
        save_checkpoint(checkpoint)
        native_ok += 1
        print(f'✅ {pdf} | {len(raw):,} chars | {time.time()-t0:.1f}s')
    except Exception as e:
        native_fail.append(pdf)
        print(f'❌ {pdf} | {e}')

print(f'\nNative done: {native_ok}/{len(native_pdfs)}')
if native_fail:
    print('Failed:', native_fail)

BƯỚC 2 — NATIVE EXTRACT (2 file)
⏭️  SKIP 623781605-The-Muscle-and-Strength-Training-Pyramid-v2-0-Training-by-Eric-Helms-Z-lib-org-1.pdf (đã có output)
⏭️  SKIP The Muscle and Strength Pyramid.pdf (đã có output)

Native done: 2/2


## 4️⃣ Bước 3 — Scanned OCR Setup

Khởi tạo PaddleOCR với wrapper **tương thích cả 2.x và 3.x**.
Ưu tiên GPU; fallback sang Tesseract nếu cần.

In [6]:
import paddle
import paddleocr
import pytesseract
import numpy as np
from paddleocr import PaddleOCR

# ── Kiểm tra GPU ──────────────────────────────────────────────────────────
GPU_OK = paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
PADDLE_MAJOR = int(str(paddleocr.__version__).split('.')[0])

print(f'Paddle version   : {paddle.__version__}')
print(f'PaddleOCR version: {paddleocr.__version__} (major={PADDLE_MAJOR})')
print(f'GPU ready        : {GPU_OK}')
if GPU_OK:
    print(f'GPU count        : {paddle.device.cuda.device_count()}')

# ── Khởi tạo engine với fallback ─────────────────────────────────────────
def _try_init(lang: str, use_gpu: bool) -> PaddleOCR | None:
    """Thử khởi tạo PaddleOCR với API 2.x hoặc 3.x."""
    attempts = [
        {'lang': lang, 'use_gpu': use_gpu, 'use_angle_cls': False, 'show_log': False},
        {'lang': lang, 'use_gpu': use_gpu, 'show_log': False},
        {'lang': lang, 'use_gpu': use_gpu},
        {'lang': lang},
    ]
    for kwargs in attempts:
        try:
            engine = PaddleOCR(**kwargs)
            print(f'  ✅ PaddleOCR OK — lang={lang}, gpu={use_gpu}, kwargs={list(kwargs.keys())}')
            return engine
        except Exception as e:
            print(f'  ⚠️  {kwargs} → {e}')
    return None

def build_ocr_state() -> dict:
    # Thử lần lượt: vi GPU → latin GPU → en GPU → vi CPU → tesseract
    for lang in [OCR_LANG_PRIMARY, OCR_LANG_FALLBACK, 'en']:
        for use_gpu in ([True, False] if GPU_OK else [False]):
            print(f'⏳ Thử khởi tạo PaddleOCR lang={lang}, gpu={use_gpu}...')
            eng = _try_init(lang, use_gpu)
            if eng is not None:
                return {'backend': 'paddle', 'engine': eng, 'lang': lang, 'gpu': use_gpu}

    print('⚠️  PaddleOCR không khởi tạo được → dùng Tesseract fallback')
    return {'backend': 'tesseract', 'engine': None, 'lang': TESSERACT_LANG, 'gpu': False}

OCR_STATE = build_ocr_state()
print(f"\n🚀 Backend đang dùng: {OCR_STATE['backend']} | lang={OCR_STATE['lang']} | gpu={OCR_STATE['gpu']}")

/usr/local/lib/python3.12/dist-packages/paddle/base/framework.py:688: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
/tmp/ipykernel_7163/229570121.py:28: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  engine = PaddleOCR(**kwargs)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Paddle version   : 2.6.2
PaddleOCR version: 3.4.0 (major=3)
GPU ready        : False
⏳ Thử khởi tạo PaddleOCR lang=vi, gpu=False...
  ⚠️  {'lang': 'vi', 'use_gpu': False, 'use_angle_cls': False, 'show_log': False} → Unknown argument: show_log
  ⚠️  {'lang': 'vi', 'use_gpu': False, 'show_log': False} → Unknown argument: show_log
  ⚠️  {'lang': 'vi', 'use_gpu': False} → Unknown argument: use_gpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


  ⚠️  {'lang': 'vi'} → 'paddle.base.libpaddle.AnalysisConfig' object has no attribute 'set_optimization_level'
⏳ Thử khởi tạo PaddleOCR lang=latin, gpu=False...
  ⚠️  {'lang': 'latin', 'use_gpu': False, 'use_angle_cls': False, 'show_log': False} → No models are available for the language 'latin' and OCR version None.
  ⚠️  {'lang': 'latin', 'use_gpu': False, 'show_log': False} → No models are available for the language 'latin' and OCR version None.
  ⚠️  {'lang': 'latin', 'use_gpu': False} → No models are available for the language 'latin' and OCR version None.
  ⚠️  {'lang': 'latin'} → No models are available for the language 'latin' and OCR version None.
⏳ Thử khởi tạo PaddleOCR lang=en, gpu=False...
  ⚠️  {'lang': 'en', 'use_gpu': False, 'use_angle_cls': False, 'show_log': False} → Unknown argument: show_log
  ⚠️  {'lang': 'en', 'use_gpu': False, 'show_log': False} → Unknown argument: show_log
  ⚠️  {'lang': 'en', 'use_gpu': False} → Unknown argument: use_gpu
  ⚠️  {'lang': 'en'} → 

### Bước 3 — OCR từng file (RAM-safe)

In [8]:
from pdf2image import convert_from_path

# ── Unified OCR wrapper: tương thích PaddleOCR 2.x và 3.x ────────────────
def _paddle_ocr(engine, img_np: np.ndarray) -> str:
    """Gọi engine.ocr() và parse kết quả cho cả 2.x lẫn 3.x."""
    try:
        result = engine.ocr(img_np, cls=False)
    except TypeError:
        result = engine.ocr(img_np)

    if not result:
        return ''

    lines = []
    # PaddleOCR 2.x: result = [ [ [bbox, (text, conf)], ... ] ]
    # PaddleOCR 3.x: result = [ OCRResult ] hoặc tương tự 2.x
    data = result[0] if isinstance(result, list) else result
    if data is None:
        return ''

    for item in data:
        if item is None:
            continue
        try:
            # 3.x: object có .text attribute
            if hasattr(item, 'text'):
                txt = item.text
            # 2.x / 3.x list format: [bbox, (text, conf)]
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                tc = item[1]
                txt = tc[0] if isinstance(tc, (list, tuple)) else str(tc)
            else:
                txt = str(item)
            if txt and str(txt).strip():
                lines.append(str(txt).strip())
        except Exception:
            continue
    return '\n'.join(lines)

def ocr_page(pil_img, state: dict) -> str:
    """OCR một trang ảnh — chọn backend tự động."""
    if state['backend'] == 'tesseract':
        return pytesseract.image_to_string(pil_img, lang=state['lang'], config='--psm 6')

    rgb = np.array(pil_img)
    # Đảm bảo ảnh là RGB 3 kênh
    if rgb.ndim == 2:
        rgb = np.stack([rgb, rgb, rgb], axis=-1)
    elif rgb.shape[2] == 4:
        rgb = rgb[:, :, :3]
    bgr = rgb[:, :, ::-1].copy()  # PaddleOCR dùng BGR

    try:
        return _paddle_ocr(state['engine'], bgr)
    except Exception as e:
        if state['backend'] == 'paddle':
            print(f'  ⚠️  Paddle lỗi trang ({e}), fallback Tesseract')
            return pytesseract.image_to_string(pil_img, lang=TESSERACT_LANG, config='--psm 6')
        raise

def process_scanned_pdf(pdf_path: str, state: dict) -> str:
    """
    OCR toàn bộ PDF scan.
    RAM-safe: render từng batch nhỏ, xóa ảnh ngay sau OCR.
    """
    doc = fitz.open(pdf_path)
    total = len(doc)
    doc.close()

    if MAX_PAGES_PER_FILE:
        total = min(total, MAX_PAGES_PER_FILE)

    page_texts = []
    t_start = time.time()

    for start in range(1, total + 1, OCR_BATCH_PAGES):
        end = min(start + OCR_BATCH_PAGES - 1, total)

        # Render batch → ảnh PIL
        imgs = convert_from_path(
            pdf_path,
            dpi=OCR_DPI,
            first_page=start,
            last_page=end,
            thread_count=4,
        )

        for i, img in enumerate(imgs):
            page_no = start + i
            txt = ocr_page(img, state)

            # ── Giải phóng ảnh ngay lập tức ──────────────────────────────
            try:
                img.close()
            except Exception:
                pass
            del img

            txt = re.sub(r'[ \t]+', ' ', txt)
            txt = re.sub(r'\n{3,}', '\n\n', txt)
            page_texts.append(txt.strip() if txt.strip() else f'[TRANG_TRỐNG_{page_no}]')

            if page_no % 10 == 0 or page_no == total:
                elapsed = max(time.time() - t_start, 1e-6)
                print(f'   📄 Trang {page_no}/{total} | {page_no/elapsed:.2f} trang/s')

        del imgs
        gc.collect()  # Dọn RAM sau mỗi batch

    return '\n\n'.join(page_texts)

# ── Chạy OCR ─────────────────────────────────────────────────────────────
scanned_ok, scanned_fail = 0, []
print('=' * 55)
print(f'BƯỚC 3 — SCANNED OCR ({len(scanned_pdfs)} file)')
print('=' * 55)

for pdf in scanned_pdfs:
    out_path = os.path.join(BOOKS_OUT, f'{Path(pdf).stem}.txt')

    if SKIP_IF_TXT_EXISTS and os.path.isfile(out_path):
        print(f'⏭️  SKIP {pdf} (đã có output)')
        scanned_ok += 1
        continue

    t0 = time.time()
    print(f'\n🔍 OCR: {pdf}')
    try:
        raw = process_scanned_pdf(os.path.join(BOOKS_DIR, pdf), OCR_STATE)
        if not raw.strip():
            raise ValueError('Không OCR được text')
        lang = detect_lang(raw)
        write_output(pdf, raw, f'Scanned OCR ({OCR_STATE["backend"]} {OCR_STATE["lang"]})', lang)
        checkpoint[pdf] = {'status': 'ocr_ok', 'chars': len(raw), 'backend': OCR_STATE['backend']}
        save_checkpoint(checkpoint)
        scanned_ok += 1
        elapsed = time.time() - t0
        print(f'✅ {pdf} | {len(raw):,} chars | {elapsed/60:.1f} min')
    except Exception as e:
        scanned_fail.append(pdf)
        checkpoint[pdf] = {'status': 'ocr_fail', 'error': str(e)}
        save_checkpoint(checkpoint)
        print(f'❌ {pdf} | {e}')
    finally:
        gc.collect()  # Dọn sau mỗi file, dù thành công hay thất bại

print(f'\nScanned OCR done: {scanned_ok}/{len(scanned_pdfs)}')
if scanned_fail:
    print('Failed:', scanned_fail)

BƯỚC 3 — SCANNED OCR (5 file)
⏭️  SKIP ACSM Guidelines for Exercise Testing and Prescription.pdf (đã có output)
⏭️  SKIP Back Mechanic - Stuart McGill - Director's Cut Edition-1.pdf (đã có output)

🔍 OCR: Science and Development of Muscle Hypertrophy — Brad Schoenfeld.pdf
   📄 Trang 10/313 | 0.15 trang/s
   📄 Trang 20/313 | 0.10 trang/s
   📄 Trang 30/313 | 0.09 trang/s
   📄 Trang 40/313 | 0.08 trang/s
   📄 Trang 50/313 | 0.08 trang/s
   📄 Trang 60/313 | 0.08 trang/s
   📄 Trang 70/313 | 0.08 trang/s
   📄 Trang 80/313 | 0.08 trang/s
   📄 Trang 90/313 | 0.08 trang/s
   📄 Trang 100/313 | 0.08 trang/s
   📄 Trang 110/313 | 0.08 trang/s
   📄 Trang 120/313 | 0.08 trang/s
   📄 Trang 130/313 | 0.08 trang/s
   📄 Trang 140/313 | 0.08 trang/s
   📄 Trang 150/313 | 0.07 trang/s
   📄 Trang 160/313 | 0.07 trang/s
   📄 Trang 170/313 | 0.08 trang/s
   📄 Trang 180/313 | 0.07 trang/s
   📄 Trang 190/313 | 0.07 trang/s
   📄 Trang 200/313 | 0.07 trang/s
   📄 Trang 210/313 | 0.07 trang/s
   📄 Trang 220/313 | 0

## 5️⃣ Bước 4 — Post-processing & Chuẩn hóa

Chạy **sau khi** Bước 2 & 3 hoàn tất. Áp dụng cho tất cả file `.txt` trong `BOOKS_OUT`.

- **Loại rác**: header/footer số trang, ký tự lạ, watermark
- **Line-unbreak**: nối các dòng bị ngắt dở câu (rất quan trọng cho Embedding models)
- **Unicode normalize**: chuẩn hóa NFC, sửa lỗi encoding mojibake
- **Encoding**: ghi lại UTF-8

In [9]:
import glob
import unicodedata

# ── Regex patterns ────────────────────────────────────────────────────────
_RE_CONTROL  = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')   # control chars
_RE_SPACES   = re.compile(r'[ \t]+')                                 # khoảng trắng thừa
_RE_BLANK3   = re.compile(r'\n{3,}')                                 # quá nhiều dòng trống
_RE_PAGE_NUM = re.compile(r'^\s*\d{1,4}\s*$', re.MULTILINE)        # số trang đứng một mình
_RE_MOJIBAKE = re.compile(r'[\xc0-\xff][\x80-\xbf]+')               # sequence UTF-8 sai

def clean_text(text: str) -> str:
    """Làm sạch toàn diện: loại rác, chuẩn hóa khoảng trắng, fix encoding."""
    # 1. Unicode normalize NFC (đặc biệt quan trọng cho tiếng Việt)
    text = unicodedata.normalize('NFC', text)
    # 2. Loại control characters
    text = _RE_CONTROL.sub('', text)
    # 3. Loại số trang đứng một mình
    text = _RE_PAGE_NUM.sub('', text)
    # 4. Chuẩn hóa khoảng trắng
    text = _RE_SPACES.sub(' ', text)
    # 5. Giảm dòng trống liên tiếp
    text = _RE_BLANK3.sub('\n\n', text)
    return text.strip()

def line_unbreak(text: str) -> str:
    """
    Nối các dòng bị ngắt dở câu.
    Logic: nếu dòng hiện tại không kết thúc bằng dấu câu KẾT THÚC
    (. ! ? : ; — ...) → nối với dòng tiếp theo bằng khoảng trắng.
    Dòng trống vẫn giữ nguyên để phân đoạn.
    """
    END_PUNCT = re.compile(r'[.!?:;\—\-]\s*$')
    lines = text.split('\n')
    merged, buf = [], ''

    for line in lines:
        stripped = line.strip()
        if not stripped:                     # Dòng trống → đẩy buffer, tạo đoạn mới
            if buf:
                merged.append(buf)
                buf = ''
            merged.append('')
        elif END_PUNCT.search(stripped):     # Dòng kết thúc câu → flush
            buf = (buf + ' ' + stripped).strip() if buf else stripped
            merged.append(buf)
            buf = ''
        else:                                # Dòng dở câu → tích vào buffer
            buf = (buf + ' ' + stripped).strip() if buf else stripped

    if buf:  # Flush buffer cuối
        merged.append(buf)

    return '\n'.join(merged)

# ── Áp dụng cho tất cả file output ───────────────────────────────────────
txt_files = glob.glob(os.path.join(BOOKS_OUT, '*.txt'))
print(f'BƯỚC 4 — POST-PROCESSING ({len(txt_files)} file)')
print('=' * 55)

for fpath in txt_files:
    try:
        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            content = f.read()

        # Tách header (những dòng bắt đầu bằng '# ') khỏi body
        header_lines, body_lines = [], []
        in_header = True
        for line in content.split('\n'):
            if in_header and (line.startswith('# ') or line == ''):
                header_lines.append(line)
            else:
                in_header = False
                body_lines.append(line)

        body = '\n'.join(body_lines)
        body = clean_text(body)
        body = line_unbreak(body)

        final = '\n'.join(header_lines) + '\n' + body

        # Ghi lại UTF-8 (bắt buộc có encoding='utf-8')
        with open(fpath, 'w', encoding='utf-8') as f:
            f.write(final)

        print(f'✅ {os.path.basename(fpath)} | {len(body):,} chars')
    except Exception as e:
        print(f'❌ {os.path.basename(fpath)} | {e}')

print(f'\n✅ Post-processing xong {len(txt_files)} file.')

BƯỚC 4 — POST-PROCESSING (7 file)
✅ 623781605-The-Muscle-and-Strength-Training-Pyramid-v2-0-Training-by-Eric-Helms-Z-lib-org-1.txt | 556,157 chars
✅ The Muscle and Strength Pyramid.txt | 522,663 chars
✅ ACSM Guidelines for Exercise Testing and Prescription.txt | 1,297,959 chars
✅ Back Mechanic - Stuart McGill - Director's Cut Edition-1.txt | 401,350 chars
✅ Science and Development of Muscle Hypertrophy — Brad Schoenfeld.txt | 1,254,723 chars
✅ Starting Strength — Mark Rippetoe.txt | 467,869 chars
✅ The Muscle and Strength Pyramid 1 .txt | 643,555 chars

✅ Post-processing xong 7 file.


## 6️⃣ Bước 5 — Kiểm tra Output

Xem nhanh kết quả và thống kê tổng thể.

In [ ]:
import glob

txt_files = sorted(glob.glob(os.path.join(BOOKS_OUT, '*.txt')))
total_chars = 0

print('=' * 55)
print(f'BƯỚC 5 — OUTPUT SUMMARY ({len(txt_files)} file)')
print('=' * 55)

for fpath in txt_files:
    size_kb = os.path.getsize(fpath) / 1024
    with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
        n_chars = len(f.read())
    total_chars += n_chars
    print(f'📄 {os.path.basename(fpath):40s} | {size_kb:7.1f} KB | {n_chars:>10,} chars')

print('-' * 55)
print(f'TỔNG: {len(txt_files)} file | {total_chars:,} chars ({total_chars/1e6:.2f}M)')
print()

# ── Xem nội dung file đầu tiên ────────────────────────────────────────────
if txt_files:
    sample = txt_files[0]
    print(f'📖 Preview: {os.path.basename(sample)}')
    print('-' * 55)
    with open(sample, 'r', encoding='utf-8') as f:
        print(f.read()[:800])
    print('\n... [ĐÃ CẮT BỚT] ...')
else:
    print('⚠️  Chưa có file output nào được tạo.')